In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv('GroqAPIKey')

In [15]:
%pip install ipython

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from groq import Groq
from openai import OpenAI
import json
from IPython.display import Markdown,display


### Agents Response - for same question - in different hits

In [26]:
client = Groq(api_key=api_key)

content = "I am planning to buy a course for Agentic AI. Give me top 2 courses that i can buy. The price should be below Rs. 1000. Just give me course name, provider and price, that's all. No extra info needed."

for i in range (0,3):
    resp = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role":"user",
                "content":content
            }
        ]
    )

    ##print(f"Response from hit - {i} : \n{resp.choices[0].message.content}")
    print(f"Response from hit - {i}")
    display(Markdown(resp.choices[0].message.content))


Response from hit - 0


Agentic AI: Building Autonomous Agents – Udemy – ≈ Rs 799  
Introduction to Agentic AI – Skillshare – ≈ Rs 999

Response from hit - 1


- Introduction to Agentic AI – Udemy – ₹799  
- Agentic AI Fundamentals – Coursera – ₹950

Response from hit - 2


- Agentic AI: Build Autonomous AI Agents with OpenAI API – Udemy – ₹749  
- Practical Agentic AI with LangChain – Udemy – ₹899

#### We can see that the agents are giving different responses for the same question. This is because the agents do not have a proper out structure provided.
- We can tell the agent to give the response in a structured format like JSON or XML. This will help the agent to give the response in a structured format and we can easily parse the response and use it in our application.

In [52]:
content = "I am planning to buy a course for Agentic AI. Give me top 2 courses that i can buy. The price should be below Rs. 1000. Just give me course name, provider and price, that's all. No extra info needed."

resp = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        response_format={"type":"json_object"},
        messages=[
            {
                "role":"system",
                "content":"Extract the response, and present it in a json format. The key tags should be Course Name, Price, Vendor."
            },{
                "role":"user",
                "content":content
            }
        ]
    )

display(Markdown(resp.choices[0].message.content))
# respDict =  json.loads(resp.choices[0].message.content)
# #print(respDict)
# for j in range (0,len(respDict)):
#     print(f"Course Name: {respDict[j]['Course Name']}")
#     print(f"Price: {respDict[j]['Price']}")
#     print(f"Vendor: {respDict[j]['Vendor']}")
    



[
{"Course Name":"Agentic AI Foundations","Price":"₹799","Vendor":"Udemy"},{"Course Name":"Introduction to Agentic AI Systems","Price":"₹999","Vendor":"Coursera"}
]

Now the o/p format has been finalised, but still we will get different responses like missing of comma, rupee symbol etc. For that we have to define the output format in a more strict way. We can use the `jsonschema` to define the output format. The `jsonschema` is a powerful tool that allows us to define the structure of the JSON data. We can use it to define the required fields, data types, and other constraints.

In [65]:
jsonSchemaDeclarationMultiple = {
    "type": "json_schema",
    "json_schema": {
        "name": "CoursesExtraction",
        "strict": True,
       "schema": {
    "type": "object",
    "properties": {
        "courses": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "Course Name": {"type": "string"},
                    "Price": {"type": "string"},
                    "Provider": {"type": "string"}
                },
                "required": [
                    "Course Name",
                    "Price",
                    "Provider"
                ],
                "additionalProperties": False
            }
        }
    },
    "required": ["courses"],
    "additionalProperties": False
}
    }
}

In [66]:
jsonSchemaDeclarationSingle = {
    "type": "json_schema",
    "json_schema": {
        "name": "CoursesExtraction",
        "strict": True,
       "schema": {
    "type": "object",
    "properties": {
        "Course Name": {"type": "string"},
        "Price": {"type": "string"},
        "Provider": {"type": "string"}
        },
        "required": [
                    "Course Name",
                    "Price",
                    "Provider"
                ],
                "additionalProperties": False
            
    }
    }
}

Here, if you see there are two types of schema, one is single and anither one is multiple. In single schema, only one result will be returned in the object format. But in case, in prompt, if i give top 2, give 5 types etc, then the agent has to give that many no.of points. So, the response will in the array format, not object. So this has to be handled in the schema as well. So, we have to define the schema in such a way that it can handle both single and multiple responses.

In [67]:
content = "I am planning to buy a course for Agentic AI. Give me top 1 course that i can buy. The price should be below Rs. 1000. Just give me course name, provider and price, that's all. No extra info needed."

resp = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        response_format=jsonSchemaDeclarationSingle,
        messages=[
            {
                "role":"system",
                "content":"Extract the response, and present it in a json format. The key tags should be Course Name, Price, Vendor."
            },{
                "role":"user",
                "content":content
            }
        ]
    )

display(Markdown(resp.choices[0].message.content))

{"Course Name":"Agentic AI Foundations","Price":"₹899","Provider":"Udemy"}